# MP1: Business Context & Data Exploration

**MajsterPlus — Customer Lapse Prediction**

In this mini-project, you will:
1. Load and verify the MajsterPlus dataset
2. Explore the structure, types, and missing values
3. Analyze the relationship between customers and transactions
4. Visualize key distributions and relationships
5. Train a quick "sneak peek" baseline model

**CRISP-DM Phase**: Business Understanding + Data Understanding

---

## 0. Setup & Reproducibility

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Library version assertions
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — expected 1.4.x or 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — expected 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — expected <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Data Loading & Verification

In [ ]:
import hashlib
from pathlib import Path

# Colab detection
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
except ImportError:
    DATA_DIR = Path("../2. data")

assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

# Load and verify customers.csv
customers = pd.read_csv(DATA_DIR / "customers.csv")
cust_md5 = hashlib.md5((DATA_DIR / "customers.csv").read_bytes()).hexdigest()
assert cust_md5 == "c016cd4bfc36ac8d0a99bb6a3d55fa44", (
    f"customers.csv MD5 mismatch: {cust_md5} != c016cd4bfc36ac8d0a99bb6a3d55fa44"
)
print(f"customers.csv loaded: {customers.shape}, MD5 OK")

transactions = pd.read_csv(DATA_DIR / "transactions.csv")
print(f"Transactions shape: {transactions.shape}")

# Load and verify transactions.csv
txn_md5 = hashlib.md5((DATA_DIR / "transactions.csv").read_bytes()).hexdigest()
assert txn_md5 == "a9c4bcfc4878baae6f5c250d4d15d737", (
    f"transactions.csv MD5 mismatch: {txn_md5} != a9c4bcfc4878baae6f5c250d4d15d737"
)
print(f"transactions.csv verified: MD5 OK")

## 2. First Look at the Data

In [ ]:
# Shape and column overview
print(f"Customers: {customers.shape[0]} rows, {customers.shape[1]} columns")
print(f"\nColumn names:\n{list(customers.columns)}")

In [ ]:
# Data types
customers.dtypes

In [ ]:
# First 5 rows
customers.head()

In [ ]:
# Statistical summary of numeric columns
customers.describe()

## 2b. Dataset Relationship Analysis

**Task**: You have two datasets: `customers.csv` and `transactions.csv`. Before diving into individual columns, think about how these datasets relate to each other.

1. Compare the shapes of both datasets
2. Compute the average number of transactions per customer
3. Interpret what this ratio tells you about purchase frequency

*Hint: How many transactions does each customer have on average? What does this tell you about purchase frequency?*

In [ ]:
# YOUR CODE HERE
# Compare the shapes of customers and transactions
# Calculate the average number of transactions per customer
# Print your findings and interpretation

## 3. Missing Value Analysis

**Task**: Analyze missing values across all columns.

1. Identify which columns have missing data and what percentage is missing in each
2. Display your results as a sorted table
3. **Interpretation**: Which column's missingness is most concerning for modeling? Consider: which has the highest missing rate? Could that column's missingness be related to the target variable (MAR — Missing At Random)?

In [ ]:
# YOUR CODE HERE
# Analyze missing values — which columns have them? What percentage is missing?
# Display results as a sorted table (descending by missing count)

In [ ]:
# YOUR CODE HERE
# Interpretation: Which column's missingness is most concerning and why?
# Consider MAR (Missing At Random) — could the missingness pattern be
# related to purchasing behavior or the target variable?
# Write your interpretation as print() statements or comments

## 4. Target Variable & Class Imbalance

In [ ]:
# Target distribution
target_dist = customers["is_lapsed"].value_counts(normalize=True)
print("Target distribution:")
print(target_dist)
print(f"\nLapse rate: {target_dist[1]:.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
customers["is_lapsed"].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title("Target Variable Distribution (is_lapsed)")
ax.set_xlabel("Is Lapsed")
ax.set_ylabel("Count")
ax.set_xticklabels(["Active (0)", "Lapsed (1)"], rotation=0)
plt.tight_layout()
plt.show()

**Task**: Think about the implications of this class distribution.

If you built a naive classifier that predicted ALL customers as "active" (`is_lapsed=0`), what accuracy would it achieve? Compute this from the target distribution above.

*Hint: Think about what percentage of the data is class 0...*

In [ ]:
# YOUR CODE HERE
# Compute the accuracy of a naive "predict all as active" classifier
# What does this tell you about using accuracy as a metric on imbalanced data?

## 5. Correlation Heatmap

**Task**: Create a correlation heatmap of all numeric columns. What strong correlations do you notice? Which features might be problematic for modeling?

In [ ]:
# YOUR CODE HERE
# Create a correlation matrix of numeric columns and visualize it as a heatmap
# Annotate the heatmap with correlation values
# Identify pairs with |r| > 0.5

## 6. Key Relationship Plots

**Task**: Create visualizations exploring the relationship between features and the target variable `is_lapsed`. Focus on:
- `days_since_last_purchase` by `is_lapsed`
- `purchase_count` by `is_lapsed`
- `avg_basket_value` distribution — do you spot any outliers?

In [ ]:
# YOUR CODE HERE
# Create 3 visualizations: two comparing features by lapse status,
# and one showing the distribution of avg_basket_value
# Arrange them side by side using subplots

**Task**: You found extreme values in `avg_basket_value`. Interpret your findings:
- Are these likely valid data points or data entry errors?
- How might they affect model training (think about linear models vs. tree-based models)?

In [ ]:
# YOUR CODE HERE
# Identify outliers in avg_basket_value
# Print their values and your interpretation
# Consider: data entry error or valid? Impact on different model types?

## 7. Sneak Peek Baseline Model

Let's train a quick LogisticRegression on raw numeric features to see how
learnable this problem is. We'll use only clean numeric columns (no preprocessing).
This gives us a baseline to beat in later mini-projects.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

# Select clean numeric features (no text, no missing-heavy columns)
# Exclude days_since_last_purchase — it's quasi-deterministic with the target
baseline_features = [
    "purchase_count", "avg_basket_value", "product_categories_bought",
    "customer_service_contacts", "store_distance_km", "account_age_days",
]

X_raw = customers[baseline_features].copy()
# Remove basket value outliers for fair baseline
mask = X_raw["avg_basket_value"] < 5000
X_raw = X_raw[mask]
y_raw = customers.loc[X_raw.index, "is_lapsed"]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)

y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]

print(f"Baseline LogisticRegression (6 raw numeric features)")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nCan we beat this with proper preprocessing and feature engineering?")

## 8. Key Observations

**Task**: Summarize your findings from this exploration. Respond to each of the following prompts:

1. **Dataset relationship**: What did the ratio of transactions to customers tell you?
2. **Class imbalance**: What is the lapse rate, and what does the naive classifier accuracy imply about using accuracy as a metric?
3. **Missing values**: Which column's missingness is most concerning and why?
4. **Data quality issues**: List all the data quality problems you discovered (think: types, formats, outliers, impossible values)
5. **Baseline performance**: What ROC-AUC did the baseline achieve, and what does it tell you about learnability?

In [ ]:
# YOUR CODE HERE
# Write your observations as print statements addressing each of the 5 prompts above
# Example:
# print("1. Dataset relationship: ...")
# print("2. Class imbalance: ...")
# print("3. Missing values: ...")
# print("4. Data quality issues: ...")
# print("5. Baseline performance: ...")

---
*End of MP1 — Continue to MP2: Data Cleaning & Feature Engineering*